In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


运行前输出配置
- 当前项目根
- 配置文件路径
- 当前 config_hash
- seed、阈值规则、调参指标、bootstrap 次数
- purchase_month/hour/dayofweek 的归属说明
- E1/E2/E3 特征数

In [1]:
from pathlib import Path
import json
import hashlib

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / "outputs" / "experiment_config.json").exists():
            return p
    raise FileNotFoundError(
        "找不到 outputs/experiment_config.json，请确认项目根目录"
    )

ROOT = find_project_root()
CONFIG_PATH = ROOT / "outputs" / "experiment_config.json"

config_text = CONFIG_PATH.read_text(encoding="utf-8")
config = json.loads(config_text)
config_hash = hashlib.md5(CONFIG_PATH.read_bytes()).hexdigest()

print("ROOT =", ROOT)
print("CONFIG_PATH =", CONFIG_PATH)
print("config_hash =", config_hash)
print("seed =", config["seed"])
print("label =", config["label"])
print("split =", config["split"])
print("id =", config["id"])
print("stages =", config["stages"])
print("tune_metric =", config["tune_metric"])
print("threshold_rule =", config["threshold_rule"])
print("class_weight_options =", config["class_weight_options"])
print("bootstrap_n =", config["bootstrap_n"])
print("notes:")
for n in config.get("notes", []):
    print(" -", n)

# 顺便加载特征规格，确认路径正确
with open(ROOT / config["feature_spec"], encoding="utf-8") as f:
    spec = json.load(f)

print("feature_spec stages:", [k for k in spec.keys() if k in ("E1", "E2", "E3")])
print("E1 特征数:", len(spec["E1"]))
print("E2 特征数:", len(spec["E2"]))
print("E3 特征数:", len(spec["E3"]))

ROOT = d:\课程\5001\group\msbd5001-olist
CONFIG_PATH = d:\课程\5001\group\msbd5001-olist\outputs\experiment_config.json
config_hash = b87da4113d0531db6784a1c61e906ced
seed = 42
label = low_rating
split = split
id = order_id
stages = ['E1', 'E2', 'E3']
tune_metric = average_precision
threshold_rule = 验证集 F1 最大；同分取较高阈值
class_weight_options = [None, 'balanced']
bootstrap_n = 1000
notes:
 - 阈值只在验证集选择，绝不在 test 上挑阈值。
 - SMOTE 不使用。
 - E1/E2/E3 使用同一批 order_id 与同一 split。
 - purchase_month / purchase_hour / purchase_dayofweek 的归属：暂按 B 的 02_features.json 版本归入 E1，并累计进入 E2/E3。
 - 此项为待 A 和 C 组会确认的暂定决策；若组会改为归 E3，必须同步修改 02_features.json 并重跑所有实验。
 - 旧 metrics 保留旧 config_hash；新实验只使用最新 config_hash。
feature_spec stages: ['E1', 'E2', 'E3']
E1 特征数: 5
E2 特征数: 18
E3 特征数: 25


列分类：把特征分成数值 / 类别 / 二元

In [2]:
from pathlib import Path
import json

ROOT = Path.cwd()
while not (ROOT / "outputs" / "experiment_config.json").exists():
    ROOT = ROOT.parent

with open(ROOT / "data/processed/02_features.json", encoding="utf-8") as f:
    spec = json.load(f)

CATEGORICAL_COLS = {
    "customer_state", "customer_city",
    "purchase_month", "purchase_hour", "purchase_dayofweek",
}
BINARY_COLS = {
    "has_credit_card", "has_voucher",
    "late_delivery", "multi_seller", "invalid_price",
}

for stage in ["E1", "E2", "E3"]:
    cols = spec[stage]
    num = [c for c in cols if c not in CATEGORICAL_COLS | BINARY_COLS]
    cat = [c for c in cols if c in CATEGORICAL_COLS]
    bin_ = [c for c in cols if c in BINARY_COLS]
    print(stage, "num:", len(num), "cat:", len(cat), "bin:", len(bin_))
    print("  num:", num)
    print("  cat:", cat)
    print("  bin:", bin_)

E1 num: 2 cat: 3 bin: 0
  num: ['order_approved_wait_days', 'estimated_delivery_days']
  cat: ['purchase_month', 'purchase_hour', 'purchase_dayofweek']
  bin: []
E2 num: 11 cat: 5 bin: 2
  num: ['order_approved_wait_days', 'estimated_delivery_days', 'order_item_count', 'seller_count', 'total_price', 'total_freight', 'avg_product_weight_g', 'payment_total', 'payment_count', 'max_installments', 'payment_type_count']
  cat: ['purchase_month', 'purchase_hour', 'purchase_dayofweek', 'customer_state', 'customer_city']
  bin: ['has_credit_card', 'has_voucher']
E3 num: 15 cat: 5 bin: 5
  num: ['order_approved_wait_days', 'estimated_delivery_days', 'order_item_count', 'seller_count', 'total_price', 'total_freight', 'avg_product_weight_g', 'payment_total', 'payment_count', 'max_installments', 'payment_type_count', 'delivery_duration_days', 'delivery_delay_days', 'freight_ratio', 'same_state_share']
  cat: ['purchase_month', 'purchase_hour', 'purchase_dayofweek', 'customer_state', 'customer_city'

验证预处理Pipeline

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "outputs" / "experiment_config.json").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from train_evaluate import load_stage, build_masks, make_preprocessor

for stage in ["E1", "E2", "E3"]:
    for model_name in ["logreg", "rf"]:
        X, y, split = load_stage(stage)
        masks = build_masks(split)
        pre = make_preprocessor(stage, model_name)
        X_tr = pre.fit_transform(X.loc[masks["train"]])
        X_va = pre.transform(X.loc[masks["val"]])
        print(f"{stage} / {model_name}: train {X_tr.shape}, val {X_va.shape}")

E1 / logreg: train (61260, 46), val (13127, 46)
E1 / rf: train (61260, 46), val (13127, 46)
E2 / logreg: train (61260, 3689), val (13127, 3689)
E2 / rf: train (61260, 3689), val (13127, 3689)
E3 / logreg: train (61260, 3696), val (13127, 3696)
E3 / rf: train (61260, 3696), val (13127, 3696)


验证make_model:
- 确认 make_model 在 train_evaluate.py 里能被 import。
- 确认三类模型都能被正确构造。
- 确认 SEED 从 config 读出来是 42，不是硬编码。

In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "outputs" / "experiment_config.json").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

import importlib
import train_evaluate
importlib.reload(train_evaluate)

from train_evaluate import make_model, SEED

for name in ["dummy", "logreg", "rf"]:
    m = make_model(name, class_weight=None)
    print(name, "->", type(m).__name__, "| params:", m.get_params())

print("SEED =", SEED)

dummy -> DummyClassifier | params: {'constant': None, 'random_state': 42, 'strategy': 'most_frequent'}
logreg -> LogisticRegression | params: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 2000, 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
rf -> RandomForestClassifier | params: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 300, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
SEED = 42


验证函数 choose_threshold

In [4]:
from sklearn.metrics import f1_score
import numpy as np

def choose_threshold(y_val, p_val, grid):
    best_t, best_f1 = None, -1.0
    for t in grid:
        f1 = f1_score(y_val, (p_val >= t).astype(int), pos_label=1)
        if f1 > best_f1:
            best_f1, best_t = f1, t
        elif f1 == best_f1 and (best_t is None or t > best_t):
            best_t = t
    return best_t, best_f1

# 用假数据快速验证
y_fake = np.array([0, 0, 1, 1, 0, 1])
p_fake = np.array([0.1, 0.4, 0.35, 0.8, 0.2, 0.9])
t, f1 = choose_threshold(y_fake, p_fake, [0.1, 0.2, 0.3, 0.4, 0.5])
print("best threshold:", t, "best F1:", f1)

best threshold: 0.3 best F1: 0.8571428571428571


验证 compute_metrics

In [5]:
from sklearn.metrics import (
    average_precision_score,
    f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix,
)

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "AP": average_precision_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "threshold": float(threshold),
    }

# 用假数据验证
m = compute_metrics(y_fake, p_fake, 0.3)
print(m)
cm = confusion_matrix(y_fake, (p_fake >= 0.3).astype(int), labels=[0, 1])
print("confusion matrix:\n", cm)

{'AP': 0.9166666666666665, 'F1': 0.8571428571428571, 'precision': 0.75, 'recall': 1.0, 'ROC_AUC': 0.888888888888889, 'threshold': 0.3}
confusion matrix:
 [[2 1]
 [0 3]]


验证 append_metrics

In [6]:
import pandas as pd
from pathlib import Path

def append_metrics(metrics_path: Path, record: dict) -> None:
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    df_new = pd.DataFrame([record])

    if metrics_path.exists():
        df_old = pd.read_csv(metrics_path)
        if "config_hash" not in df_old.columns:
            df_old["config_hash"] = "legacy_no_hash"
        df_all = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_all = df_new

    cols = list(df_all.columns)
    if "config_hash" in cols:
        cols.insert(0, cols.pop(cols.index("config_hash")))
        df_all = df_all[cols]

    df_all.to_csv(metrics_path, index=False)

# 用临时路径验证，不要污染正式 metrics.csv
tmp_path = ROOT / "outputs" / "_tmp_test_metrics.csv"
if tmp_path.exists():
    tmp_path.unlink()

append_metrics(tmp_path, {"experiment": "E1", "model": "logreg", "AP": 0.5, "config_hash": "testhash"})
append_metrics(tmp_path, {"experiment": "E2", "model": "rf", "AP": 0.6, "config_hash": "testhash"})

df = pd.read_csv(tmp_path)
print(df)
print("columns:", df.columns.tolist())

# 清理
tmp_path.unlink()
print("append_metrics 检查通过。")

  config_hash experiment   model   AP
0    testhash         E1  logreg  0.5
1    testhash         E2      rf  0.6
columns: ['config_hash', 'experiment', 'model', 'AP']
append_metrics 检查通过。


验证 run_experiment

In [ ]:
import importlib
import train_evaluate
importlib.reload(train_evaluate)
from train_evaluate import run_experiment

# 先只跑 E1 + LogReg，验证流程走通
rec, p_va, p_te, y_va, y_te = run_experiment("E1", "logreg", class_weight=None, tag="E1 feasibility")


[E1/logreg] val AP=0.1820 F1=0.2850 t=0.15 | test AP=0.1771 F1=0.2581


补 E1 class_weight=balanced,LogReg 和 RF 各跑一次，看平衡权重是否让 F1/recall 改善,只在 val 上选
跑 E2/RF 和 E3/RF

In [13]:
import importlib
import train_evaluate
importlib.reload(train_evaluate)

from train_evaluate import run_experiment

# 1. 先补 E1 的 balanced 对照
run_experiment("E1", "logreg", class_weight="balanced", tag="E1 feasibility | cw=balanced")
run_experiment("E1", "rf",     class_weight="balanced", tag="E1 feasibility | cw=balanced")

# 2. 再跑 E2/E3 的 RF
run_experiment("E2", "rf", class_weight=None, tag="E2 feasibility")
run_experiment("E3", "rf", class_weight=None, tag="E3 feasibility")

[E1/logreg] val AP=0.1824 F1=0.2845 t=0.45 | test AP=0.1771 F1=0.2520
[E1/rf] val AP=0.1675 F1=0.2819 t=0.05 | test AP=0.1637 F1=0.2639
[E2/rf] val AP=0.2359 F1=0.2820 t=0.05 | test AP=0.2254 F1=0.2642
[E3/rf] val AP=0.2557 F1=0.3012 t=0.20 | test AP=0.2342 F1=0.2678


({'experiment': 'E3',
  'model': 'rf',
  'class_weight': None,
  'n_train': 61260,
  'n_val': 13127,
  'n_test': 13127,
  'AP': 0.2341839745900752,
  'F1': 0.26778115501519756,
  'precision': 0.19235807860262008,
  'recall': 0.4405,
  'ROC_AUC': 0.580261323806956,
  'threshold': 0.2,
  'val_AP': 0.25568452988836304,
  'val_F1': 0.3011954665424624,
  'seed': 42,
  'config_hash': 'b87da4113d0531db6784a1c61e906ced',
  'notes': 'E3 feasibility'},
 array([0.25333333, 0.23      , 0.43666667, ..., 0.21333333, 0.24666667,
        0.15333333], shape=(13127,)),
 array([0.19333333, 0.07333333, 0.17333333, ..., 0.21      , 0.15666667,
        0.20666667], shape=(13127,)),
 11       0
 14       0
 22       1
 24       0
 27       1
         ..
 87481    0
 87482    0
 87492    0
 87498    0
 87501    0
 Name: low_rating, Length: 13127, dtype: int64,
 1        0
 2        0
 12       0
 21       0
 30       0
         ..
 87474    0
 87475    0
 87484    0
 87488    0
 87502    0
 Name: low_rating, 

E2/logreg、E3/logreg 各跑一次。
E2/logreg、E2/rf、E3/logreg、E3/rf 的 balanced 版各跑一次。
总共新增 6 行到 metrics.csv。

In [3]:
import importlib
import train_evaluate
importlib.reload(train_evaluate)
from train_evaluate import run_experiment

# 2.1 补 E2/E3 LogReg
rec1, *_ = run_experiment("E2", "logreg", class_weight=None, tag="E2 main")
rec2, *_ = run_experiment("E3", "logreg", class_weight=None, tag="E3 main")

for stage in ["E2", "E3"]:
    for model in ["logreg", "rf"]:
        rec, *_ = run_experiment(
            stage, model,
            class_weight="balanced",
            tag=f"{stage} main | cw=balanced"
        )

[E2/logreg] val AP=0.2456 F1=0.2912 t=0.15 | test AP=0.2333 F1=0.2694
[E3/logreg] val AP=0.2652 F1=0.3085 t=0.15 | test AP=0.2547 F1=0.2961
[E2/logreg] val AP=0.2447 F1=0.2911 t=0.45 | test AP=0.2311 F1=0.2683
[E2/rf] val AP=0.2388 F1=0.2840 t=0.20 | test AP=0.2282 F1=0.2671
[E3/logreg] val AP=0.2650 F1=0.3146 t=0.50 | test AP=0.2522 F1=0.2761
[E3/rf] val AP=0.2584 F1=0.3107 t=0.30 | test AP=0.2465 F1=0.2796


小网格调参

In [3]:
from train_evaluate import tune_params

tuning_results = []
for stage in ["E1", "E2", "E3"]:
    for model in ["logreg", "rf"]:
        best_params, best_ap = tune_params(stage, model, class_weight=None)
        tuning_results.append({
            "stage": stage, "model": model,
            "best_val_ap": best_ap, "best_params": best_params,
        })

import pandas as pd
pd.DataFrame(tuning_results).to_csv(
    ROOT / "outputs" / "04_tuning_results.csv", index=False
)

[tune] E1/logreg: best val AP=0.1824, params=(0.1, 'balanced')
[tune] E1/rf: best val AP=0.1774, params=(500, 10, 5, None)
[tune] E2/logreg: best val AP=0.2520, params=(0.1, None)
[tune] E2/rf: best val AP=0.2571, params=(300, 10, 5, 'balanced')
[tune] E3/logreg: best val AP=0.2720, params=(0.1, None)
[tune] E3/rf: best val AP=0.2736, params=(300, 10, 1, None)


用调好的参数再跑一次 run_experiment 得到最终 test 指标

In [2]:
import importlib
import train_evaluate
importlib.reload(train_evaluate)
from train_evaluate import run_experiment

# E1/logreg
run_experiment("E1", "logreg", class_weight="balanced",
               params={"C": 0.1}, tag="E1 final | tuned")

# E1/rf
run_experiment("E1", "rf", class_weight=None,
               params={"n_estimators": 500, "max_depth": 10, "min_samples_leaf": 5},
               tag="E1 final | tuned")

# E2/logreg
run_experiment("E2", "logreg", class_weight=None,
               params={"C": 0.1}, tag="E2 final | tuned")

# E2/rf
run_experiment("E2", "rf", class_weight="balanced",
               params={"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 5},
               tag="E2 final | tuned")

# E3/logreg
run_experiment("E3", "logreg", class_weight=None,
               params={"C": 0.1}, tag="E3 final | tuned")

# E3/rf
run_experiment("E3", "rf", class_weight=None,
               params={"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 1},
               tag="E3 final | tuned")

[E1/logreg] val AP=0.1824 F1=0.2840 t=0.45 | test AP=0.1773 F1=0.2526
[E1/rf] val AP=0.1774 F1=0.2821 t=0.15 | test AP=0.1746 F1=0.2615
[E2/logreg] val AP=0.2520 F1=0.2897 t=0.15 | test AP=0.2363 F1=0.2691
[E2/rf] val AP=0.2571 F1=0.2844 t=0.50 | test AP=0.2431 F1=0.2550
[E3/logreg] val AP=0.2720 F1=0.3137 t=0.20 | test AP=0.2586 F1=0.2576
[E3/rf] val AP=0.2736 F1=0.2842 t=0.20 | test AP=0.2504 F1=0.2559


({'experiment': 'E3',
  'model': 'rf',
  'class_weight': None,
  'params': "{'n_estimators': 300, 'max_depth': 10, 'min_samples_leaf': 1}",
  'n_train': 61260,
  'n_val': 13127,
  'n_test': 13127,
  'AP': 0.25038698145612026,
  'F1': 0.25585480093676816,
  'precision': 0.3086158192090395,
  'recall': 0.2185,
  'ROC_AUC': 0.5844296980318144,
  'threshold': 0.2,
  'val_AP': 0.2736110191310373,
  'val_F1': 0.2842025699168556,
  'seed': 42,
  'config_hash': 'b87da4113d0531db6784a1c61e906ced',
  'notes': 'E3 final | tuned'},
 array([0.18131226, 0.1761447 , 0.33835404, ..., 0.17402592, 0.17672377,
        0.16653143], shape=(13127,)),
 array([0.18162329, 0.16749202, 0.16733463, ..., 0.17360469, 0.16620386,
        0.16875494], shape=(13127,)),
 11       0
 14       0
 22       1
 24       0
 27       1
         ..
 87481    0
 87482    0
 87492    0
 87498    0
 87501    0
 Name: low_rating, Length: 13127, dtype: int64,
 1        0
 2        0
 12       0
 21       0
 30       0
         ..


消融实验

In [6]:
import importlib
import train_evaluate
importlib.reload(train_evaluate)
from train_evaluate import run_ablation

variants = ["E3_no_delivery", "E3_no_payment",
            "E3_no_product", "E3_no_geo"]

for v in variants:
    rec1, *_ = run_ablation(
        v, "logreg", class_weight=None,
        params={"C": 0.1},
        tag=f"{v} | ablation"
    )
    rec2, *_ = run_ablation(
        v, "rf", class_weight=None,
        params={"n_estimators": 300, "max_depth": 10,
                "min_samples_leaf": 1},
        tag=f"{v} | ablation"
    )

[E3_no_delivery/logreg] val AP=0.2512 F1=0.2893 t=0.15 | test AP=0.2368 F1=0.2726
[E3_no_delivery/rf] val AP=0.2551 F1=0.2820 t=0.15 | test AP=0.2405 F1=0.2644
[E3_no_payment/logreg] val AP=0.2709 F1=0.3132 t=0.15 | test AP=0.2575 F1=0.2960
[E3_no_payment/rf] val AP=0.2657 F1=0.2826 t=0.20 | test AP=0.2501 F1=0.2438
[E3_no_product/logreg] val AP=0.2207 F1=0.2951 t=0.15 | test AP=0.2054 F1=0.2671
[E3_no_product/rf] val AP=0.2116 F1=0.2820 t=0.15 | test AP=0.1852 F1=0.2644
[E3_no_geo/logreg] val AP=0.2705 F1=0.3166 t=0.15 | test AP=0.2574 F1=0.2946
[E3_no_geo/rf] val AP=0.2713 F1=0.3170 t=0.20 | test AP=0.2495 F1=0.2715


跑完后生成消融对比表

In [9]:
import pandas as pd

df = pd.read_csv(ROOT / "outputs" / "metrics.csv")
abl = df[df["notes"].str.contains("ablation", na=False)]

pivot = abl.pivot_table(
    index="experiment",
    columns="model",
    values=["AP", "val_AP"],
    aggfunc="max"
)
print(pivot.to_string())
pivot.to_csv(ROOT / "outputs" / "05_ablation_results.csv")

                      AP              val_AP          
model             logreg        rf    logreg        rf
experiment                                            
E3_no_delivery  0.236752  0.240489  0.251193  0.255079
E3_no_geo       0.257393  0.249481  0.270457  0.271290
E3_no_payment   0.257543  0.250077  0.270922  0.265703
E3_no_product   0.205402  0.185160  0.220746  0.211592


保存消融结果，生成 05_ablation_results.csv，再把它扩展成带差值的表：

In [12]:
import pandas as pd

df = pd.read_csv(ROOT / "outputs" / "metrics.csv")
abl = df[df["notes"].str.contains("ablation", na=False)]

pivot = abl.pivot_table(
    index="experiment", columns="model",
    values=["AP", "val_AP"], aggfunc="max"
)

# 压平 MultiIndex 列：AP_logreg, AP_rf, val_AP_logreg, val_AP_rf
pivot.columns = [f"{metric}_{model}" for metric, model in pivot.columns]

# E3 full 基准
e3_full = df[df["notes"].str.contains("final | tuned", na=False)]
e3_full = e3_full[e3_full["experiment"].eq("E3")]
e3_ap = e3_full.set_index("model")["AP"].to_dict()
print("E3 full:", e3_ap)

# 计算差值
delta = pd.DataFrame(index=pivot.index)
for model in ["logreg", "rf"]:
    delta[f"AP_{model}_delta"] = pivot[f"AP_{model}"] - e3_ap[model]

# 合并
combined = pivot.join(delta)
print(combined.to_string())
combined.to_csv(ROOT / "outputs" / "05_ablation_results.csv")
print("已写入 05_ablation_results.csv")


E3 full: {'logreg': 0.2585521602724477, 'rf': 0.2503869814561202}
                AP_logreg     AP_rf  val_AP_logreg  val_AP_rf  AP_logreg_delta  AP_rf_delta
experiment                                                                                 
E3_no_delivery   0.236752  0.240489       0.251193   0.255079        -0.021801    -0.009898
E3_no_geo        0.257393  0.249481       0.270457   0.271290        -0.001159    -0.000906
E3_no_payment    0.257543  0.250077       0.270922   0.265703        -0.001009    -0.000310
E3_no_product    0.205402  0.185160       0.220746   0.211592        -0.053150    -0.065227
已写入 05_ablation_results.csv


配对 bootstrap，量化 E2−E1、E3−E2 的 AP 差值区间。

重跑主实验和消融

In [15]:
import importlib
import train_evaluate
importlib.reload(train_evaluate)
from train_evaluate import run_experiment, run_ablation

# 重跑 E1/E2/E3 tuned（让预测落盘）
run_experiment("E1", "logreg", class_weight="balanced",
               params={"C": 0.1}, tag="E1 final | tuned")
run_experiment("E2", "logreg", class_weight=None,
               params={"C": 0.1}, tag="E2 final | tuned")
run_experiment("E3", "logreg", class_weight=None,
               params={"C": 0.1}, tag="E3 final | tuned")

# 如果也要 RF 的 bootstrap，把 RF 一起跑
run_experiment("E1", "rf", class_weight=None,
               params={"n_estimators": 500, "max_depth": 10, "min_samples_leaf": 5},
               tag="E1 final | tuned")
run_experiment("E2", "rf", class_weight="balanced",
               params={"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 5},
               tag="E2 final | tuned")
run_experiment("E3", "rf", class_weight=None,
               params={"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 1},
               tag="E3 final | tuned")

# 消融也要重跑一遍，或者只重跑 LogReg 的四个 variant
for v in ["E3_no_delivery", "E3_no_payment", "E3_no_product", "E3_no_geo"]:
    rec, *_ = run_ablation(v, "logreg", class_weight=None,
                           params={"C": 0.1}, tag=f"{v} | ablation")

[E1/logreg] val AP=0.1824 F1=0.2840 t=0.45 | test AP=0.1773 F1=0.2526
[E2/logreg] val AP=0.2520 F1=0.2897 t=0.15 | test AP=0.2363 F1=0.2691
[E3/logreg] val AP=0.2720 F1=0.3137 t=0.20 | test AP=0.2586 F1=0.2576
[E1/rf] val AP=0.1774 F1=0.2821 t=0.15 | test AP=0.1746 F1=0.2615
[E2/rf] val AP=0.2571 F1=0.2844 t=0.50 | test AP=0.2431 F1=0.2550
[E3/rf] val AP=0.2736 F1=0.2842 t=0.20 | test AP=0.2504 F1=0.2559
[E3_no_delivery/logreg] val AP=0.2512 F1=0.2893 t=0.15 | test AP=0.2368 F1=0.2726
[E3_no_payment/logreg] val AP=0.2709 F1=0.3132 t=0.15 | test AP=0.2575 F1=0.2960
[E3_no_product/logreg] val AP=0.2207 F1=0.2951 t=0.15 | test AP=0.2054 F1=0.2671
[E3_no_geo/logreg] val AP=0.2705 F1=0.3166 t=0.15 | test AP=0.2574 F1=0.2946


In [16]:
import pandas as pd

metrics_path = ROOT / "outputs" / "metrics.csv"
df = pd.read_csv(metrics_path)

print("去重前行数:", len(df))
print(df[["experiment", "model", "class_weight", "notes"]].value_counts().to_string())

# 去重键：同 config_hash + 同 experiment + 同 model + 同 class_weight + 同 notes = 同一次实验
key = ["config_hash", "experiment", "model", "class_weight", "notes"]
df_clean = df.drop_duplicates(subset=key, keep="last").reset_index(drop=True)

# 保证 config_hash 第一列
cols = list(df_clean.columns)
cols.insert(0, cols.pop(cols.index("config_hash")))
df_clean = df_clean[cols]

df_clean.to_csv(metrics_path, index=False)

print("\n去重后行数:", len(df_clean))
print(df_clean[["config_hash", "experiment", "model", "class_weight", "notes", "AP", "F1"]].to_string())

去重前行数: 27
experiment  model   class_weight  notes                       
E1          logreg  balanced      E1 feasibility | cw=balanced    1
            rf      balanced      E1 feasibility | cw=balanced    1
E2          logreg  balanced      E2 main | cw=balanced           1
            rf      balanced      E2 main | cw=balanced           1
E3          logreg  balanced      E3 main | cw=balanced           1
            rf      balanced      E3 main | cw=balanced           1
E1          logreg  balanced      E1 final | tuned                1
E2          rf      balanced      E2 final | tuned                1

去重后行数: 27
                         config_hash      experiment   model class_weight                         notes        AP        F1
0   b87da4113d0531db6784a1c61e906ced              E1   dummy          NaN                   E0 baseline  0.152358  0.000000
1   b87da4113d0531db6784a1c61e906ced              E1  logreg          NaN                E1 feasibility  0.177137  0.258109


检查 test_predictions.csv

In [17]:
import pandas as pd
pred = pd.read_csv(ROOT / "outputs" / "test_predictions.csv")
print(pred.shape)
print(pred["experiment"].value_counts())
print(pred["tag"].value_counts())
print(pred.head())

(131270, 6)
experiment
E1                26254
E2                26254
E3                26254
E3_no_delivery    13127
E3_no_payment     13127
E3_no_product     13127
E3_no_geo         13127
Name: count, dtype: int64
tag
E1 final | tuned             26254
E2 final | tuned             26254
E3 final | tuned             26254
E3_no_delivery | ablation    13127
E3_no_payment | ablation     13127
E3_no_product | ablation     13127
E3_no_geo | ablation         13127
Name: count, dtype: int64
   order_id  y_true    y_prob experiment   model               tag
0         1       0  0.463480         E1  logreg  E1 final | tuned
1         2       0  0.427893         E1  logreg  E1 final | tuned
2        12       0  0.422408         E1  logreg  E1 final | tuned
3        21       0  0.380381         E1  logreg  E1 final | tuned
4        30       0  0.410800         E1  logreg  E1 final | tuned


In [18]:
import pandas as pd
base = pd.read_parquet(ROOT / "data/processed/base_orders.parquet", columns=["order_id"])
print(base["order_id"].head())
print(base["order_id"].dtype)

0    e481f51cbdc54678b7cc49136f2d6af7
1    53cdb2fc8bc7dce0b6741e2150273451
2    47770eb9100c2d0c44946d9cf07ec65d
3    949d5b44dbf5de918fe9c16f97b45f8a
4    ad21c59c0840e6cb83a9ceb5573f8159
Name: order_id, dtype: str
str


In [19]:
import pandas as pd
pred = pd.read_csv(ROOT / "outputs" / "test_predictions.csv")
print("dtype:", pred["order_id"].dtype)
print(pred["order_id"].head(10).to_string())

dtype: int64
0     1
1     2
2    12
3    21
4    30
5    37
6    41
7    44
8    45
9    53


重跑保存预测，重跑主实验的 LogReg 三组 + 消融 LogReg 四组

In [21]:
from pathlib import Path
old = ROOT / "outputs" / "test_predictions.csv"
if old.exists():
    old.unlink()

import importlib
import train_evaluate
importlib.reload(train_evaluate)
from train_evaluate import run_experiment, run_ablation

for stage, cw in [("E1", "balanced"), ("E2", None), ("E3", None)]:
    rec, *_ = run_experiment(stage, "logreg", class_weight=cw,
                             params={"C": 0.1}, tag=f"{stage} final | tuned")

for v in ["E3_no_delivery", "E3_no_payment", "E3_no_product", "E3_no_geo"]:
    rec, *_ = run_ablation(v, "logreg", class_weight=None,
                           params={"C": 0.1}, tag=f"{v} | ablation")

[E1/logreg] val AP=0.1824 F1=0.2840 t=0.45 | test AP=0.1773 F1=0.2526
[E2/logreg] val AP=0.2520 F1=0.2897 t=0.15 | test AP=0.2363 F1=0.2691
[E3/logreg] val AP=0.2720 F1=0.3137 t=0.20 | test AP=0.2586 F1=0.2576
[E3_no_delivery/logreg] val AP=0.2512 F1=0.2893 t=0.15 | test AP=0.2368 F1=0.2726
[E3_no_payment/logreg] val AP=0.2709 F1=0.3132 t=0.15 | test AP=0.2575 F1=0.2960
[E3_no_product/logreg] val AP=0.2207 F1=0.2951 t=0.15 | test AP=0.2054 F1=0.2671
[E3_no_geo/logreg] val AP=0.2705 F1=0.3166 t=0.15 | test AP=0.2574 F1=0.2946


In [22]:
import pandas as pd
pred = pd.read_csv(ROOT / "outputs" / "test_predictions.csv")
print("dtype:", pred["order_id"].dtype)
print(pred["order_id"].head().to_string())
print("shape:", pred.shape)

dtype: str
0    53cdb2fc8bc7dce0b6741e2150273451
1    47770eb9100c2d0c44946d9cf07ec65d
2    5ff96c15d0b717ac6ad1f3d77225a350
3    f3e7c359154d965827355f39d6b1fdac
4    b276e4f8c0fb86bd82fce576f21713e0
shape: (91889, 6)


跑 bootstrap

In [25]:
import importlib
import train_evaluate
importlib.reload(train_evaluate)
from train_evaluate import paired_bootstrap

import pandas as pd

pred = pd.read_csv(ROOT / "outputs" / "test_predictions.csv")

def get_pred(exp, model, tag_contains):
    sub = pred[(pred["experiment"].eq(exp)) &
               (pred["model"].eq(model)) &
               (pred["tag"].str.contains(tag_contains, na=False))]
    sub = sub.sort_values("order_id").reset_index(drop=True)
    return sub["y_true"].values, sub["y_prob"].values

# 先确认不同实验的 test 订单完全对齐
y1, _ = get_pred("E1", "logreg", "final | tuned")
y2, _ = get_pred("E2", "logreg", "final | tuned")
y3, _ = get_pred("E3", "logreg", "final | tuned")
assert (y1 == y2).all() and (y2 == y3).all(), "test 集不一致"
print("y_true 对齐检查通过, n =", len(y1))

_, p_e1 = get_pred("E1", "logreg", "final | tuned")
_, p_e2 = get_pred("E2", "logreg", "final | tuned")
_, p_e3 = get_pred("E3", "logreg", "final | tuned")

results = []
for name, p_a, p_b in [
    ("E2 - E1", p_e2, p_e1),
    ("E3 - E2", p_e3, p_e2),
    ("E3 - E1", p_e3, p_e1),
]:
    m, ci = paired_bootstrap(y1, p_a, p_b, n=1000, seed=42)
    results.append({
        "model": "logreg",
        "comparison": name,
        "mean": m,
        "ci_low": ci[0],
        "ci_high": ci[1],
    })
    print(f"logreg {name}: mean={m:+.4f}, 95% CI=[{ci[0]:+.4f}, {ci[1]:+.4f}]")

out = pd.DataFrame(results)
out.to_csv(ROOT / "outputs" / "05_bootstrap_results.csv", index=False)
print("\n已写入 05_bootstrap_results.csv")
print(out.to_string(index=False))

y_true 对齐检查通过, n = 13127
logreg E2 - E1: mean=+0.0589, 95% CI=[+0.0450, +0.0739]
logreg E3 - E2: mean=+0.0223, 95% CI=[+0.0151, +0.0292]
logreg E3 - E1: mean=+0.0811, 95% CI=[+0.0661, +0.0972]

已写入 05_bootstrap_results.csv
 model comparison     mean   ci_low  ci_high
logreg    E2 - E1 0.058881 0.045050 0.073936
logreg    E3 - E2 0.022259 0.015112 0.029215
logreg    E3 - E1 0.081141 0.066063 0.097160


跑消融的 bootstrap

In [26]:
y3, p_e3 = get_pred("E3", "logreg", "final | tuned")

abl_results = []
for v in ["E3_no_delivery", "E3_no_payment", "E3_no_product", "E3_no_geo"]:
    _, p_v = get_pred(v, "logreg", "ablation")
    m, ci = paired_bootstrap(y3, p_e3, p_v, n=1000, seed=42)
    abl_results.append({
        "variant": v, "mean": m,
        "ci_low": ci[0], "ci_high": ci[1],
    })
    print(f"E3 - {v}: mean={m:+.4f}, 95% CI=[{ci[0]:+.4f}, {ci[1]:+.4f}]")

pd.DataFrame(abl_results).to_csv(
    ROOT / "outputs" / "05_bootstrap_ablation.csv", index=False
)

E3 - E3_no_delivery: mean=+0.0218, 95% CI=[+0.0148, +0.0287]
E3 - E3_no_payment: mean=+0.0010, 95% CI=[-0.0000, +0.0020]
E3 - E3_no_product: mean=+0.0527, 95% CI=[+0.0388, +0.0674]
E3 - E3_no_geo: mean=+0.0012, 95% CI=[-0.0030, +0.0054]
